# 📊 EDA — EN↔VI Translation Dataset
> Exploratory Data Analysis cho dự án AI Translator sử dụng **Qwen 2.5**

---

In [ ]:
# ============================================================
# 📦 IMPORTS & CONFIG
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re
import warnings
warnings.filterwarnings('ignore')

# --- Style ---
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d2e',
    'axes.edgecolor':   '#3d4166',
    'axes.labelcolor':  '#c8cce8',
    'xtick.color':      '#8b8fb5',
    'ytick.color':      '#8b8fb5',
    'text.color':       '#c8cce8',
    'grid.color':       '#2a2d45',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   14,
    'axes.labelsize':   12,
})
PALETTE = ['#7c6af7', '#f7a26a', '#5ecec6', '#f06292', '#aed581']
print('✅ Imports done!')

✅ Imports done!


In [ ]:
!pip install underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.7 MB/s eta 0:00:00


In [ ]:
from underthesea import word_tokenize
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

en_stopwords = set(stopwords.words('english'))
print(list(en_stopwords)[:20])

["isn't", 're', 'those', "hadn't", 'other', "shan't", "we'll", 'because', "he'd", "i'll", 'my', 'our', "shouldn't", 'above', 'weren', 'that', 'their', 'theirs', 'i', 'll']


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


## 1️⃣ Load dữ liệu
> Thay `PATH` bằng đường dẫn file CSV của bạn.

In [ ]:
# ============================================================
# 📂 LOAD DATA — đổi PATH ở đây
# ============================================================
PATH = '/content/dataset_en_vi_all_domains_filtered_v3 (1).csv'   # <-- đổi thành đường dẫn thực tế

df = pd.read_csv(PATH)

# Auto-detect cột en / vi (case-insensitive)
col_map = {c.lower(): c for c in df.columns}
EN_COL = col_map.get('en', df.columns[0])
VI_COL = col_map.get('vi', df.columns[1])

print(f'📌 Cột EN : {EN_COL}')
print(f'📌 Cột VI : {VI_COL}')
print(f'📐 Shape  : {df.shape}')
df.head(5)

📌 Cột EN : en
📌 Cột VI : vi
📐 Shape  : (102616, 3)


,en,vi,domain
0,"3PL contracts are long term contracts, whereas...","Hợp đồng 3PL là hợp đồng dài hạn, trong khi hợ...",economic
1,They took Ruth while she was out buying food.,Họ bắt Ruth khi cô ấy đi mua thức ăn.,general
2,"Um, "" said Margo. I thought about shoving off ...","Ơ, "" Margo lên tiếng. Tôi toan té và phóng thẳ...",general
3,Coordinating with the Ministry of Finance in s...,Phối hợp với Bộ Tài chính xác định và công bố ...,economic
4,"And I think, in the future, journalism and man...","Và tôi nghĩ, trong tương lai, báo chí và nhiều...",general


## 2️⃣ Tổng quan dữ liệu

In [ ]:
# ============================================================
# 📋 BASIC STATS
# ============================================================
total = len(df)
missing_en = df[EN_COL].isna().sum()
missing_vi = df[VI_COL].isna().sum()
dupes       = df.duplicated(subset=[EN_COL, VI_COL]).sum()
empty_en    = (df[EN_COL].astype(str).str.strip() == '').sum()
empty_vi    = (df[VI_COL].astype(str).str.strip() == '').sum()

stats = pd.DataFrame({
    'Metric':  ['Tổng cặp câu', 'Missing EN', 'Missing VI',
                'Dòng trùng lặp', 'Chuỗi rỗng EN', 'Chuỗi rỗng VI'],
    'Count':   [total, missing_en, missing_vi, dupes, empty_en, empty_vi],
    'Tỉ lệ %': [100,
                round(missing_en/total*100,2),
                round(missing_vi/total*100,2),
                round(dupes/total*100,2),
                round(empty_en/total*100,2),
                round(empty_vi/total*100,2)]
})

display(stats.style
    .background_gradient(subset='Count', cmap='RdYlGn_r')
    .format({'Tỉ lệ %': '{:.2f}%'})
    .set_caption('📊 Data Quality Overview')
)

,Metric,Count,Tỉ lệ %
0,Tổng cặp câu,102616,100.00%
1,Missing EN,0,0.00%
2,Missing VI,0,0.00%
3,Dòng trùng lặp,3071,2.99%
4,Chuỗi rỗng EN,0,0.00%
5,Chuỗi rỗng VI,0,0.00%


## 3️⃣ Feature Engineering — độ dài & token

In [ ]:
# ============================================================
# 🔢 FEATURE ENGINEERING
# ============================================================
df = df.dropna(subset=[EN_COL, VI_COL]).copy()
df[EN_COL] = df[EN_COL].astype(str).str.strip()
df[VI_COL] = df[VI_COL].astype(str).str.strip()

# Độ dài ký tự
df['en_char_len'] = df[EN_COL].str.len()
df['vi_char_len'] = df[VI_COL].str.len()

# Số từ (word-level token thô)
df['en_word_cnt'] = df[EN_COL].str.split().str.len()
df['vi_word_cnt'] = df[VI_COL].apply(word_tokenize).apply(len)

AttributeError: 'Series' object has no attribute 'len'

In [ ]:
df['vi_word_cnt'] = df['vi_word_cnt'].apply(len)

In [ ]:
# Tỉ lệ độ dài VI/EN
df['len_ratio_vi_en'] = df['vi_char_len'] / (df['en_char_len'] + 1e-5)

# Ước tính BPE token (~4 chars/token)
df['en_token_est'] = (df['en_char_len'] / 4).astype(int)
df['vi_token_est'] = (df['vi_char_len'] / 4).astype(int)

print('✅ Features created!')
df[['en_char_len','vi_char_len','en_word_cnt','vi_word_cnt','len_ratio_vi_en']].describe().round(2)

✅ Features created!


,en_char_len,vi_char_len,en_word_cnt,vi_word_cnt,len_ratio_vi_en
count,102616.00,102616.00,102616.00,102616.00,102616.00
mean,255.74,243.45,40.42,43.51,0.97
std,156.12,144.17,24.02,25.59,0.17
min,14.00,9.00,2.00,1.00,0.19
25%,136.00,133.00,22.00,24.00,0.86
50%,214.00,207.00,34.00,37.00,0.96
75%,345.00,325.00,54.00,57.00,1.07
max,900.00,778.00,128.00,152.00,2.63


## 4️⃣ Phân phối độ dài câu

In [ ]:
# ============================================================
# 📈 DISTRIBUTION: CHAR LENGTH
# ============================================================
fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Độ dài ký tự — EN', 'Độ dài ký tự — VI'))

for i, (col, name, color) in enumerate([
    ('en_char_len', 'English', '#7c6af7'),
    ('vi_char_len', 'Vietnamese', '#f7a26a')
], 1):
    fig.add_trace(go.Histogram(
        x=df[col], nbinsx=60, name=name,
        marker_color=color, opacity=0.85
    ), row=1, col=i)

fig.update_layout(
    title='📏 Phân phối độ dài ký tự EN vs VI',
    template='plotly_dark', showlegend=False,
    height=380, paper_bgcolor='#0f1117', plot_bgcolor='#1a1d2e'
)
fig.show()

In [ ]:
# ============================================================
# 📈 DISTRIBUTION: WORD COUNT
# ============================================================
fig = px.histogram(
    df.melt(value_vars=['en_word_cnt','vi_word_cnt'],
            var_name='lang', value_name='word_count'),
    x='word_count', color='lang', barmode='overlay',
    nbins=60, opacity=0.75,
    color_discrete_map={'en_word_cnt': '#7c6af7', 'vi_word_cnt': '#f7a26a'},
    labels={'word_count': 'Số từ', 'lang': 'Ngôn ngữ'},
    title='📝 Phân phối số từ EN vs VI'
)
fig.update_layout(
    template='plotly_dark', height=380,
    paper_bgcolor='#0f1117', plot_bgcolor='#1a1d2e',
    legend=dict(x=0.75, y=0.95)
)
fig.show()

## 5️⃣ Tỉ lệ độ dài VI/EN — Alignment check

In [ ]:
# ============================================================
# 🔄 LENGTH RATIO DISTRIBUTION
# ============================================================
fig = px.histogram(
    df, x='len_ratio_vi_en', nbins=80,
    title='⚖️ Tỉ lệ độ dài VI/EN (lý tưởng ≈ 1.0–1.5)',
    color_discrete_sequence=['#5ecec6']
)
fig.add_vline(x=1.0, line_dash='dash', line_color='#f06292',
              annotation_text='Ratio=1.0', annotation_position='top right')
fig.add_vline(x=df['len_ratio_vi_en'].median(), line_dash='dot',
              line_color='#aed581',
              annotation_text=f'Median={df["len_ratio_vi_en"].median():.2f}',
              annotation_position='top left')
fig.update_layout(
    template='plotly_dark', height=380,
    paper_bgcolor='#0f1117', plot_bgcolor='#1a1d2e',
    xaxis_range=[0, 5]
)
fig.show()

# Các cặp có tỉ lệ bất thường
outliers = df[(df['len_ratio_vi_en'] < 0.3) | (df['len_ratio_vi_en'] > 4)]
print(f'⚠️  Cặp câu bất thường (ratio <0.3 hoặc >4): {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)')

⚠️  Cặp câu bất thường (ratio <0.3 hoặc >4): 14 (0.01%)


## 6️⃣ Scatter: EN vs VI length

In [ ]:
# ============================================================
# 🔵 SCATTER: EN char vs VI char
# ============================================================
sample = df.sample(min(3000, len(df)), random_state=42)

fig = px.scatter(
    sample, x='en_char_len', y='vi_char_len',
    color='len_ratio_vi_en',
    color_continuous_scale='Viridis',
    opacity=0.5, size_max=4,
    labels={'en_char_len': 'Độ dài EN (chars)', 'vi_char_len': 'Độ dài VI (chars)'},
    title='🔵 Tương quan độ dài EN ↔ VI'
)
# Đường tham chiếu y=x
max_val = max(sample['en_char_len'].max(), sample['vi_char_len'].max())
fig.add_trace(go.Scatter(
    x=[0, max_val], y=[0, max_val],
    mode='lines', line=dict(color='#f06292', dash='dash'),
    name='y = x'
))
fig.update_layout(
    template='plotly_dark', height=450,
    paper_bgcolor='#0f1117', plot_bgcolor='#1a1d2e'
)
fig.show()

corr = df['en_char_len'].corr(df['vi_char_len'])
print(f'📐 Pearson correlation EN↔VI char length: {corr:.4f}')

📐 Pearson correlation EN↔VI char length: 0.9545


## 9️⃣ Top N-grams (EN & VI)

In [ ]:
# ============================================================
# 🔠 TOP UNIGRAMS
# ============================================================
from collections import Counter

STOPWORDS_EN = set(stopwords.words('english'))
STOPWORDS_EN.add("shall")
STOPWORDS_VI = {'là','và','của','trong','có','một','các','được','cho','với','không','này','đã','những'}

def top_words(series, stopwords, n=20):
    tokens = series.str.lower().str.split().explode()
    tokens = tokens[~tokens.isin(stopwords)]
    tokens = tokens[tokens.str.match(r'^[a-zA-ZÀ-ỹ]+$')]
    return pd.DataFrame(Counter(tokens).most_common(n), columns=['word','count'])

top_en = top_words(df[EN_COL], STOPWORDS_EN)
top_vi = top_words(df[VI_COL], STOPWORDS_VI)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Top 20 từ EN (bỏ stopwords)', 'Top 20 từ VI (bỏ stopwords)'))

fig.add_trace(go.Bar(
    y=top_en['word'][::-1], x=top_en['count'][::-1],
    orientation='h', marker_color='#7c6af7', name='EN'
), row=1, col=1)
fig.add_trace(go.Bar(
    y=top_vi['word'][::-1], x=top_vi['count'][::-1],
    orientation='h', marker_color='#f7a26a', name='VI'
), row=1, col=2)

fig.update_layout(
    title='🔠 Top 20 từ xuất hiện nhiều nhất',
    template='plotly_dark', height=520, showlegend=False,
    paper_bgcolor='#0f1117', plot_bgcolor='#1a1d2e'
)
fig.show()